In [ ]:
!pip install pandas 

## Chunking Pipeline

This notebook converts the scraped Ooredoo dataset into smaller semantic chunks that can be embedded and indexed in ChromaDB.

The chunking pipeline:
1. Loads the merged scraped dataset.
2. Cleans noisy text and fixes encoding problems.
3. Detects page types and semantic sections.
4. Splits long text into token-based chunks.
5. Keeps structured pages such as products, listings, categories, and roaming data in suitable chunk formats.
6. Adds metadata such as URL, title, language, section, page type, `chunk_index`, `chunk_id`, and `content_hash`.
7. Saves the final chunks into a JSONL file.

In [ ]:

import hashlib
import json
import re
import unicodedata
from pathlib import Path
from typing import Dict, List
from collections import defaultdict

from transformers import AutoTokenizer


# ---------- CONFIG ----------
INPUT_PATH = Path(r"C:\Users\USER\Desktop\pfe_26\scrape_data\semantic_dataset_merged.json")
OUTPUT_JSONL = Path(r"C:\Users\USER\Desktop\pfe_26\data_chunking\semantic_chunks_generic_fixed.jsonl")

TOKENIZER_NAME = "intfloat/multilingual-e5-base"

TARGET_TOKENS = 260
OVERLAP_TOKENS = 30
SINGLE_CHUNK_MAX_TOKENS = 160


# ---------- RULES ----------
HEADING_HINTS = {
    "presentation", "présentation", "assistance", "description", "descriptif",
    "service", "services", "tarifs", "prices", "pricing", "activation",
    "désactivation", "deactivation", "faq", "avantages", "advantages",
    "technical specifics", "screen", "camera", "memory", "autonomy", "sim",
    "connection", "ergonomics", "ergonomie", "autonomie", "appareil photo / caméra",
    "spécificités techniques", "réseau", "network compatibility",
}

NOISE_LINES = {
    "back to top", "quick view", "shop now", "sort by:", "select", "relevance",
    "name, a to z", "name, z to a", "price, low to high", "price, high to low",
    "retour en haut", "j’en profite", "ok", "stock unavailable", "stock available",
}

SINGLE_CHUNK_TYPES = {
    "product_page",
    "roaming_tariff_country",
}

SEMANTIC_CHUNK_TYPES = {
    "content_page", "shop_page", "other"
}

GROUPED_LIST_TYPES = {"brand_page", "listing_page"}
NAV_TYPES = {"category_page"}

HEADING_WORDS = [
    "presentation", "présentation", "services", "service", "assistance",
    "description", "descriptif", "faq", "tarifs", "tarif", "prices", "pricing",
    "activation", "deactivation", "désactivation", "advantages", "avantages"
]
HEADING_ALT = "|".join(map(re.escape, HEADING_WORDS))


# ---------- HELPERS ----------
def maybe_fix_mojibake(text: str) -> str:
    if not text:
        return text
    if any(m in text for m in ("Ã", "Â", "â", "î")):
        try:
            repaired = text.encode("latin1", errors="ignore").decode("utf-8", errors="ignore")
            if repaired and repaired.count("Ã") < text.count("Ã"):
                return repaired
        except Exception:
            pass
    return text


def strip_heading_boilerplate(text: str) -> str:
    t = re.sub(
        rf"^(?:(?:{HEADING_ALT})\s+){{1,8}}",
        "",
        text,
        flags=re.IGNORECASE,
    ).strip()

    t = re.sub(
        rf"([.!?]\s+)(?:(?:{HEADING_ALT})\s+){{1,8}}(?=[A-ZÀ-Ý])",
        r"\1",
        t,
        flags=re.IGNORECASE,
    ).strip()

    return t


def clean_lines(text: str) -> str:
    text = maybe_fix_mojibake(text)
    text = strip_heading_boilerplate(text)
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    cleaned = []
    for raw in text.split("\n"):
        line = re.sub(r"\s+", " ", raw).strip()
        if not line:
            continue
        low = line.lower()
        if low in NOISE_LINES:
            continue
        if re.fullmatch(r"[\W_]+", line):
            continue
        cleaned.append(line)

    return "\n".join(cleaned)


def is_heading(line: str) -> bool:
    low = line.lower().strip()

    if low in HEADING_HINTS:
        return True
    if re.match(r"^article\s+\d+", low):
        return True
    if re.match(r"^boutique\s+", low):
        return True
    if low.endswith(":") and len(line.split()) <= 8:
        return True

    if len(line) <= 40 and 1 <= len(line.split()) <= 4:
        if re.match(r"^[A-Za-zÀ-ÿ /&'’+\-]+$", line) and not re.search(r"[.!?]", line):
            return True

    return False


def semantic_sections(text: str) -> List[str]:
    lines = [x.strip() for x in text.split("\n") if x.strip()]
    if not lines:
        return []

    sections: List[str] = []
    current: List[str] = []

    for line in lines:
        if is_heading(line) and current:
            sections.append("\n".join(current).strip())
            current = [line]
        else:
            current.append(line)

    if current:
        sections.append("\n".join(current).strip())

    return [s for s in sections if s]


def token_windows(
    text: str,
    tokenizer,
    target_tokens: int = 260,
    overlap_tokens: int = 30,
) -> List[str]:
    token_ids = tokenizer.encode(text, add_special_tokens=False)
    if not token_ids:
        return []
    if len(token_ids) <= target_tokens:
        return [text]

    chunks = []
    step = max(1, target_tokens - overlap_tokens)
    start = 0

    while start < len(token_ids):
        end = min(start + target_tokens, len(token_ids))
        chunk_ids = token_ids[start:end]
        chunk_text = tokenizer.decode(chunk_ids, skip_special_tokens=True).strip()
        if chunk_text:
            chunks.append(chunk_text)
        if end >= len(token_ids):
            break
        start += step

    return chunks


def infer_chunk_type(text: str, page_type: str) -> str:
    low = text.lower()

    if page_type == "product_page":
        return "product_specs"
    if page_type == "brand_page":
        return "brand_listing"
    if page_type == "listing_page":
        return "listing_group"
    if page_type == "category_page":
        return "category_nav"
    if page_type == "shop_page":
        return "shop_info"
    if page_type == "roaming_tariff_country":
        return "roaming_tariff_country"
    if page_type == "roaming_partner_operators":
        return "roaming_partner_operators"

    if "activation" in low or "souscription" in low:
        return "activation"
    if "désactivation" in low or "deactivation" in low:
        return "deactivation"
    if "tarif" in low or "pricing" in low or "prix" in low:
        return "pricing"
    if "faq" in low or "qu'est ce que" in low or "what is" in low:
        return "faq"
    if "description" in low or "descriptif" in low or "présentation" in low:
        return "description"

    return "default"


def make_chunk(
    row: Dict,
    chunk_text: str,
    chunk_index: int,
    chunk_type: str = "default",
    country: str = "",
    zone: str = "",
) -> Dict:
    chunk_hash = hashlib.sha256(
        f"{row.get('url','')}::{row.get('content_hash','')}::{chunk_type}::{country}::{zone}::{chunk_index}::{chunk_text}".encode("utf-8")
    ).hexdigest()

    out = {
        "chunk_id": chunk_hash,
        "chunk_index": chunk_index,
        "url": row.get("url"),
        "title": row.get("title"),
        "language": row.get("language"),
        "section": row.get("section"),
        "page_type": row.get("page_type", "other"),
        "content_hash": row.get("content_hash"),
        "chunk_type": chunk_type,
        "text": chunk_text,
    }

    if country:
        out["country"] = country
    if zone:
        out["zone"] = zone

    return out


def normalize_country_name(s: str) -> str:
    s = (s or "").strip().lower()
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    s = re.sub(r"[^a-z0-9]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def cleanup_partner_operator_text(source_text: str, country: str) -> str:
    """
    Keep only usable roaming partner-operator lines.
    Drop obviously malformed lines when possible.
    """
    lines = [ln.strip() for ln in source_text.split("\n") if ln.strip()]
    if not lines:
        return ""

    out = []
    normalized_country = normalize_country_name(country)

    for i, ln in enumerate(lines):
        if i == 0 and ln.lower().startswith("country:"):
            out.append(ln)
            continue

        if not ln.lower().startswith("operator:"):
            continue

        if "roaming type:" not in ln.lower():
            continue

        # Example pattern:
        # Operator: Orange | Country: France | Roaming Type: 4G
        m = re.search(r"\|\s*Country:\s*(.*?)\s*\|\s*Roaming Type:\s*(.*)$", ln, re.IGNORECASE)
        if m:
            row_country = normalize_country_name(m.group(1))
            if row_country and normalized_country and row_country != normalized_country:
                # reject clearly malformed country fragments like "sud" for "Afrique du sud"
                if row_country not in normalized_country and normalized_country not in row_country:
                    continue

        out.append(ln)

    return "\n".join(out).strip()


# ---------- SPECIAL CHUNKERS ----------
def chunk_roaming_record(row: Dict, tokenizer) -> List[Dict]:
    page_type = row.get("page_type", "other")
    source_text = (row.get("embedding_text") or row.get("clean_text") or row.get("raw_text") or "").strip()
    source_text = clean_lines(source_text)
    if not source_text:
        return []

    country = (row.get("country") or "").strip()
    zone = (row.get("zone") or "").strip()

    if page_type == "roaming_tariff_country":
        # Keep intact as one structured chunk
        return [
            make_chunk(
                row=row,
                chunk_text=source_text,
                chunk_index=0,
                chunk_type="roaming_tariff_country",
                country=country,
                zone=zone,
            )
        ]

    if page_type == "roaming_partner_operators":
        cleaned = cleanup_partner_operator_text(source_text, country)
        if not cleaned:
            cleaned = source_text

        token_ids = tokenizer.encode(cleaned, add_special_tokens=False)

        # keep as one chunk if reasonably short
        if len(token_ids) <= TARGET_TOKENS:
            return [
                make_chunk(
                    row=row,
                    chunk_text=cleaned,
                    chunk_index=0,
                    chunk_type="roaming_partner_operators",
                    country=country,
                )
            ]

        # otherwise split lightly
        chunks = []
        for i, ch in enumerate(token_windows(cleaned, tokenizer, TARGET_TOKENS, OVERLAP_TOKENS)):
            chunks.append(
                make_chunk(
                    row=row,
                    chunk_text=ch,
                    chunk_index=i,
                    chunk_type="roaming_partner_operators",
                    country=country,
                )
            )
        return chunks

    # fallback
    return [make_chunk(row=row, chunk_text=source_text, chunk_index=0, chunk_type="roaming_fallback", country=country, zone=zone)]


def chunk_grouped_listing(row: Dict) -> List[Dict]:
    source_text = (row.get("embedding_text") or row.get("clean_text") or row.get("raw_text") or "").strip()
    source_text = clean_lines(source_text)
    if not source_text:
        return []

    lines = [x.strip() for x in source_text.split("\n") if x.strip()]
    title = (row.get("title") or "").strip()

    header = []
    items = []

    for ln in lines:
        low = ln.lower()

        if title and ln.lower() == title.lower():
            continue

        if "there are" in low or "there is" in low or "il y a" in low:
            header.append(ln)
        elif "prix:" in low or "stock:" in low or "|" in ln:
            items.append(ln)
        else:
            items.append(ln)

    chunks = []
    idx = 0

    if header:
        chunks.append(make_chunk(row, "\n".join(header), idx, "listing_overview"))
        idx += 1

    batch = []
    for item in items:
        batch.append(item)
        if len(batch) == 8:
            chunks.append(make_chunk(row, "\n".join(batch), idx, "listing_group"))
            idx += 1
            batch = []

    if batch:
        chunks.append(make_chunk(row, "\n".join(batch), idx, "listing_group"))

    return chunks


def chunk_category_nav(row: Dict) -> List[Dict]:
    source_text = (row.get("embedding_text") or row.get("clean_text") or row.get("raw_text") or "").strip()
    source_text = clean_lines(source_text)
    if not source_text:
        return []

    lines = [x.strip() for x in source_text.split("\n") if x.strip()]
    if not lines:
        return []

    text = "\n".join(lines).strip()
    return [make_chunk(row, text, 0, "category_nav")]


def chunk_standard_record(
    row: Dict,
    tokenizer,
    target_tokens: int,
    overlap_tokens: int,
    single_chunk_max_tokens: int = 160,
) -> List[Dict]:
    source_text = (row.get("embedding_text") or row.get("clean_text") or row.get("raw_text") or "").strip()
    source_text = clean_lines(source_text)
    if not source_text:
        return []

    page_type = row.get("page_type", "other")
    source_tokens = len(tokenizer.encode(source_text, add_special_tokens=False))

    if page_type in SINGLE_CHUNK_TYPES and source_tokens <= single_chunk_max_tokens:
        sections = [source_text]
    elif page_type in SEMANTIC_CHUNK_TYPES:
        sections = semantic_sections(source_text)
        if not sections:
            sections = [source_text]
    else:
        sections = [source_text]

    chunk_texts: List[str] = []
    for sec in sections:
        windows = token_windows(sec, tokenizer, target_tokens, overlap_tokens)
        chunk_texts.extend(windows if windows else [sec])

    out = []
    for i, chunk in enumerate(chunk_texts):
        ctype = infer_chunk_type(chunk, page_type)
        out.append(make_chunk(row, chunk, i, ctype))
    return out


def chunk_record(
    row: Dict,
    tokenizer,
    target_tokens: int,
    overlap_tokens: int,
    single_chunk_max_tokens: int = 160,
) -> List[Dict]:
    page_type = row.get("page_type", "other")

    if page_type in {"roaming_tariff_country", "roaming_partner_operators"}:
        return chunk_roaming_record(row, tokenizer)

    if page_type in GROUPED_LIST_TYPES:
        return chunk_grouped_listing(row)

    if page_type in NAV_TYPES:
        return chunk_category_nav(row)

    return chunk_standard_record(
        row=row,
        tokenizer=tokenizer,
        target_tokens=target_tokens,
        overlap_tokens=overlap_tokens,
        single_chunk_max_tokens=single_chunk_max_tokens,
    )


def load_json(path: Path) -> List[Dict]:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def write_jsonl(path: Path, rows: List[Dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def run_pipeline() -> None:
    if not INPUT_PATH.exists():
        raise FileNotFoundError(f"Input file not found: {INPUT_PATH}")

    print("Loading dataset...")
    rows = load_json(INPUT_PATH)
    print(f"Rows: {len(rows)}")

    print("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

    print("Chunking...")
    all_chunks: List[Dict] = []
    for row in rows:
        all_chunks.extend(
            chunk_record(
                row=row,
                tokenizer=tokenizer,
                target_tokens=TARGET_TOKENS,
                overlap_tokens=OVERLAP_TOKENS,
                single_chunk_max_tokens=SINGLE_CHUNK_MAX_TOKENS,
            )
        )

    write_jsonl(OUTPUT_JSONL, all_chunks)
    print(f"Chunks generated: {len(all_chunks)}")
    print(f"Chunk file: {OUTPUT_JSONL}")


# Execute
run_pipeline()

Loading dataset...
Rows: 1451
Loading tokenizer...


Token indices sequence length is longer than the specified maximum sequence length for this model (519 > 512). Running this sequence through the model will result in indexing errors


Chunking...
Chunks generated: 2008
Chunk file: C:\Users\USER\Desktop\pfe_26\data_chunking\semantic_chunks_generic_fixed.jsonl
